In [7]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [4]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_fraud.json")

In [ ]:
# from pg_schema.loader import SchemaLoader

# schema = SchemaLoader("../../dtgraph/pg_schema/schemas/schema_fraud.json")

##### Rules

In [15]:
Rule1 = Rule('''
MATCH (c:Client)
WHERE NOT c:Mule
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 2000
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns)
})
''', env=env, type_strict=True)

Rule2 = Rule('''
MATCH (c:Client:Mule)
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 2000
GENERATE
(p = (c.id):Scammer {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns)
})
''', env=env, type_strict=True)

Rule3 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:CashIn)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_CASHIN]->((t.id):CashIn {
        original_amount = t.amount,
        formatted_amount = round(t.amount * 100) / 100.0,
        is_large_amount = t.amount > 150000
    })
''', env=env, type_strict=True)

Rule4 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:CashOut)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_CASHOUT]->(tx = (t.id):CashOut {
        original_amount = t.amount,
        formatted_amount = round(t.amount * 100) / 100.0,
        is_large_amount = t.amount > 150000
    })
''', env=env, type_strict=True)

Rule5 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Payment)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_PAYMENT]->(tx = (t.id):Payment)
''', env=env, type_strict=True)

Rule6 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Transfer)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_TRANSFER]->(tx = (t.id):Transfer)
''', env=env, type_strict=True)

Rule7 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Debit)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_DEBITS]->(tx = (t.id):Debit)
''', env=env, type_strict=True)

##### Schema Conformance

In [20]:
# Schema checking

from pg_schema.loader import SchemaLoader

schema = SchemaLoader("../../dtgraph/pg_schema/schemas/schema_fraud.json")

from dtgraph.pg_schema.check_schema import check_schema


check_schema([Rule1], schema)


--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (c:Client)\nWHERE NOT c:Mule\nOPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)\nOPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)\nOPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)\nWITH \n    c,\n    collect(DISTINCT e.email) AS emails,\n    collect(DISTINCT p.phoneNumber) AS phones,\n    collect(DISTINCT s.ssn) AS ssns\nLIMIT 2000', 'constructors': [{'alias': 'p', 'ids': ['c.id'], 'labels': ['Person'], 'properties': [{'key': 'id', 'value': 'c.id'}, {'key': 'name', 'value': 'c.name'}, {'key': 'name_camel_case', 'value': 'apoc.text.upperCamelCase(c.name)'}, {'key': 'email', 'value': 'head(emails)'}, {'key': 'phone', 'value': 'head(phones)'}, {'key': 'ssn', 'value': 'head(ssns)\n'}]}]}

RULE FAILED:
Source dictionary:
{'lhs': 'MATCH (c:Client)\nWHERE NOT c:Mule\nOPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)\nOPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)\nOPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)\nWITH \n    c,\n    collect(DISTINCT e.email) AS emails,\n    collec

CompileError: 
Schema conformance failed for one or more rules

##### Type Checking

In [ ]:
# Type checking

from dtgraph.type_checking.check_types import check_types


check_types([Rule1, Rule2, Rule3, Rule4, Rule5, Rule6, Rule7], env)

##### Applying Rules

In [ ]:
my_transform = Transformation([Rule1])
my_transform.apply_on(graph)

##### Abort Transformation

In [ ]:
my_transform.abort()